# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: C:\Users\Asus\Documents\3rd year\term 1\FDE-Manan-Gupta\nyc-tlc-cbd-pipeline\class_notebooks\flasheats


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders=orders.drop_duplicates("order_id",keep="first")
late_order=outcomes[outcomes.late_flag==1].order_id.iloc[0]
on_time_order=outcomes[outcomes.late_flag==0].order_id.iloc[0]
intervention_order=interventions.order_id.iloc[0]
print("late",late_order,"on-time",on_time_order,"intervention",intervention_order)

# TODO: build_order_timeline(order_id)

late O00001 on-time O00003 intervention O00781


In [4]:
# Dispatch data retrieved in Class 5 is kept as a raw file, so I reuse it rather than calling the API again
dispatch = pd.DataFrame(json.loads((BASE / "api" / "dispatch_data.json").read_text()))
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

def build_order_timeline(order_id):
    o = base_orders.loc[base_orders.order_id == order_id].iloc[0]
    d = dispatch.loc[dispatch.order_id == order_id].iloc[0]
    rows = [
        (o.created_at, "order_created", "customer", "orders (SQLite)"),
        (d.assigned_at, "driver_assigned", f"dispatch -> {d.original_driver_id}", "dispatch API"),
        (d.reassigned_at, "driver_reassigned", f"dispatch -> {d.driver_id}", "dispatch API"),
        (d.estimated_pickup_at, "estimated_pickup (plan)", "dispatch", "dispatch API"),
        (o.pickup_at, "picked_up", o.driver_id, "orders (SQLite)"),
        (o.promised_eta, "promised_eta (plan)", "order platform", "orders (SQLite)"),
        (o.actual_delivery_at, "delivered", o.driver_id, "orders (SQLite)"),
    ]
    for _, r in restaurant_status[restaurant_status.order_id == order_id].iterrows():
        rows.append((r.last_updated_at, f"restaurant_status: {r.status.strip().lower()}", r.restaurant_id, "restaurant_status.csv"))
    for _, r in customer_actions[customer_actions.order_id == order_id].iterrows():
        rows.append((r.action_at, r.action_type.lower(), r.customer_id, "customer_app_actions.csv"))
    for _, r in tickets[tickets.order_id == order_id].drop_duplicates("ticket_id").iterrows():
        rows.append((r.created_at, f"support_ticket: {r.category.strip()}", "customer", "support_tickets.csv"))
    for _, r in interventions[interventions.order_id == order_id].iterrows():
        rows.append((r.intervention_at, f"intervention: {r.intervention_type} ({r.reason})", r.initiated_by, "order_interventions.csv"))
    tl = pd.DataFrame(rows, columns=["event_time", "event_type", "actor", "source_system"]).dropna(subset=["event_time"])
    tl["event_time"] = pd.to_datetime(tl.event_time, format="mixed")
    return tl.sort_values("event_time").reset_index(drop=True)

for label, oid in [("ON TIME", on_time_order), ("LATE", late_order), ("WITH INTERVENTION", intervention_order)]:
    out = outcomes.loc[outcomes.order_id == oid].iloc[0]
    print(f"{label}: {oid} | outcome = {out.outcome_bucket}, delay = {out.delay_min} min")
    display(build_order_timeline(oid))

ON TIME: O00003 | outcome = delivered_on_time, delay = -4.41 min


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,order_created,customer,orders (SQLite)
1,2026-08-01 19:35:00.002443,driver_assigned,dispatch -> D039,dispatch API
2,2026-08-01 19:48:00.000000,restaurant_status: preparing,R010,restaurant_status.csv
3,2026-08-01 19:53:00.000000,estimated_pickup (plan),dispatch,dispatch API
4,2026-08-01 19:57:50.820366,picked_up,D039,orders (SQLite)
5,2026-08-01 20:13:28.061191,delivered,D039,orders (SQLite)
6,2026-08-01 20:17:52.941967,promised_eta (plan),order platform,orders (SQLite)


LATE: O00001 | outcome = delivered_late, delay = 10.82 min


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,order_created,customer,orders (SQLite)
1,2026-08-13 13:01:09.659644,driver_assigned,dispatch -> D103,dispatch API
2,2026-08-13 13:08:00.000000,intervention: DRIVER_REASSIGNMENT (driver_unavailable),dispatch,order_interventions.csv
3,2026-08-13 13:14:00.000000,estimated_pickup (plan),dispatch,dispatch API
4,2026-08-13 13:21:00.000000,eta_viewed,C0168,customer_app_actions.csv
5,2026-08-13 13:35:49.825561,picked_up,D103,orders (SQLite)
6,2026-08-13 14:11:36.000000,promised_eta (plan),order platform,orders (SQLite)
7,2026-08-13 14:20:36.000000,support_opened,C0168,customer_app_actions.csv
8,2026-08-13 14:22:25.035899,delivered,D103,orders (SQLite)


WITH INTERVENTION: O00781 | outcome = delivered_late, delay = 13.07 min


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,order_created,customer,orders (SQLite)
1,2026-08-22 12:40:42.931820,driver_assigned,dispatch -> D103,dispatch API
2,2026-08-22 12:51:00.000000,estimated_pickup (plan),dispatch,dispatch API
3,2026-08-22 12:54:00.000000,eta_viewed,C0103,customer_app_actions.csv
4,2026-08-22 12:57:00.000000,intervention: PRIORITY_DISPATCH (late_risk),operations,order_interventions.csv
5,2026-08-22 13:04:00.000000,restaurant_status: ready,R019,restaurant_status.csv
6,2026-08-22 13:11:17.309762,picked_up,D103,orders (SQLite)
7,2026-08-22 13:48:36.000000,promised_eta (plan),order platform,orders (SQLite)
8,2026-08-22 14:01:40.289785,delivered,D103,orders (SQLite)


In [5]:
# The late order has a DRIVER_REASSIGNMENT intervention, but dispatch shows the same driver throughout.
# How often do the two sources agree about reassignments?
reassign_iv = set(interventions.loc[interventions.intervention_type == "DRIVER_REASSIGNMENT", "order_id"])
reassign_dispatch = set(dispatch.loc[dispatch.reassigned_at.notna(), "order_id"])
print("Reassignment interventions logged:", len(reassign_iv))
print("Reassignments recorded by dispatch:", len(reassign_dispatch))
print("Orders where both agree:", len(reassign_iv & reassign_dispatch))

Reassignment interventions logged: 155
Reassignments recorded by dispatch: 95
Orders where both agree: 8


### Challenge 1 notes

- Putting events from six sources on one clock is what makes the workflow visible. No single source tells the whole story.
- **On time (O00003):** picked up about 5 minutes after dispatch's planned pickup, and delivered 4 minutes early.
- **Late (O00001):** picked up about 22 minutes after the planned pickup. The customer checked the ETA, then opened
  support after the promised time had passed. The delay built up before pickup, not on the road.
- **With intervention (O00781):** a priority dispatch was triggered for late risk *before* pickup, but the order was
  **still 13 minutes late**. The intervention's timing was right, but the outcome didn't change.
- **Sources disagree about reassignment.** The interventions log says 155 orders had a driver reassignment, dispatch
  records 95, and only 8 orders appear in both. The log looks like it records *requests*, and dispatch records what
  actually happened. So I treat dispatch as the system of record for reassignment.
- What's missing from every timeline: **when the driver reached the restaurant** and **when the food was actually
  ready**. Restaurant status is a single snapshot, and it doesn't exist for most orders.

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [6]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))
# TODO: document grain + relationships

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


In [7]:
def pk_check(df, key):
    return df[key].notna().all() and not df[key].duplicated().any()

def fk_cov(child, parent):
    return round(100 * child.dropna().isin(parent).mean(), 1)

model = pd.DataFrame([
    ["customer", "customer_id", "-", "one customer", pk_check(customers, "customer_id"), "-"],
    ["order", "order_id", "customer_id, restaurant_id, driver_id", "one order (after de-duplication)",
     pk_check(base_orders, "order_id"), f"customer {fk_cov(base_orders.customer_id, customers.customer_id)}%"],
    ["customer_interaction", "action_id", "order_id, customer_id", "one in-app action by a customer",
     pk_check(customer_actions, "action_id"), f"order {fk_cov(customer_actions.order_id, base_orders.order_id)}%"],
    ["support_interaction", "ticket_id", "order_id", "one support ticket",
     pk_check(tickets.drop_duplicates(), "ticket_id"), f"order {fk_cov(tickets.order_id, base_orders.order_id)}%"],
    ["intervention", "intervention_id", "order_id", "one action taken by ops, dispatch or support",
     pk_check(interventions, "intervention_id"), f"order {fk_cov(interventions.order_id, base_orders.order_id)}%"],
    ["outcome", "order_id", "order_id", "one final result per order",
     pk_check(outcomes, "order_id"), f"order {fk_cov(outcomes.order_id, base_orders.order_id)}%"],
], columns=["entity", "primary key", "foreign keys", "grain", "PK unique", "FK coverage"])
model

,entity,primary key,foreign keys,grain,PK unique,FK coverage
0,customer,customer_id,-,one customer,True,-
1,order,order_id,"customer_id, restaurant_id, driver_id",one order (after de-duplication),True,customer 100.0%
2,customer_interaction,action_id,"order_id, customer_id",one in-app action by a customer,True,order 100.0%
3,support_interaction,ticket_id,order_id,one support ticket,True,order 100.0%
4,intervention,intervention_id,order_id,"one action taken by ops, dispatch or support",True,order 100.0%
5,outcome,order_id,order_id,one final result per order,True,order 100.0%


### Challenge 2 answer

```
customer 1 ──< order 1 ──< customer_interaction   (app actions: ETA views, support opened, cancel attempts)
                     │ ──< support_interaction    (tickets)
                     │ ──< intervention           (reassignment, restaurant contact, priority dispatch, credit)
                     └── 1 outcome                (delivered / late / cancelled, delay_min)
          restaurant, driver = reference tables joined to order
```

All primary keys are unique (after dropping the one duplicated ticket row), and every foreign key that's filled in maps.

**Why this is better than mirroring the source tables:** the sources are organised by *system* (SQLite, the app,
the support tool, dispatch). The questions we need to answer are organised by *order journey*: what happened, how the
customer reacted, what we did, and how it ended. Keeping the order at the centre, with one table per kind of thing that
happens to it, means every business question becomes a join on `order_id` instead of a fresh reconciliation each time.
It's also small. Things like GPS pings or ETA model versions stay in their sources until a question needs them.

# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [8]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())
# TODO: add flags, aggregate interventions, and join to outcomes/orders

In [9]:
flags = (customer_actions.pivot_table(index="order_id", columns="action_type", values="action_id", aggfunc="count")
         .fillna(0).astype(int).reset_index())
iv = (interventions.groupby("order_id")
      .agg(intervention_count=("intervention_id", "count"),
           intervention_types=("intervention_type", lambda s: ", ".join(sorted(set(s)))))
      .reset_index())
tickets_by_order = tickets.drop_duplicates("ticket_id").dropna(subset=["order_id"]).groupby("order_id").size().rename("tickets")

order_model = (outcomes.merge(base_orders[["order_id", "customer_id", "restaurant_id"]], on="order_id", how="left")
               .merge(actions_by_order, on="order_id", how="left")
               .merge(flags, on="order_id", how="left")
               .merge(iv, on="order_id", how="left")
               .merge(tickets_by_order, on="order_id", how="left"))
num = ["action_count", "ETA_VIEWED", "SUPPORT_OPENED", "CANCEL_ATTEMPTED", "intervention_count", "tickets"]
order_model[num] = order_model[num].fillna(0).astype(int)
order_model["intervention_types"] = order_model["intervention_types"].fillna("")
order_model["support_opened"] = (order_model.SUPPORT_OPENED > 0) | (order_model.tickets > 0)
order_model["cancel_attempted"] = order_model.CANCEL_ATTEMPTED > 0
order_model = order_model.rename(columns={"final_status_norm": "final_status"})

cols = ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count", "intervention_types",
        "final_status", "late_flag", "delay_min"]
print("One row per order:", order_model.order_id.is_unique, "|", len(order_model), "rows")
display(order_model[cols].head(8))

One row per order: True | 1600 rows


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,True,False,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82
1,O00002,C0043,False,False,0,,delivered,1.0,3.97
2,O00003,C0855,False,False,0,,delivered,0.0,-4.41
3,O00004,C0229,False,False,0,,cancelled,NaN,NaN
4,O00005,C0040,False,False,0,,delivered,1.0,21.02
5,O00006,C0152,False,False,0,,delivered,1.0,7.30
6,O00007,C0875,False,False,0,,delivered,1.0,4.27
7,O00008,C0223,True,False,1,CUSTOMER_CREDIT,delivered,1.0,26.86


In [10]:
late = order_model[order_model.late_flag == 1]
frustrated = order_model[(order_model.late_flag == 1) & (order_model.support_opened | order_model.cancel_attempted)]

print("1. Late orders with a support interaction:", late.support_opened.sum(), "of", len(late),
      f"({100 * late.support_opened.mean():.0f}%)")
print("2. Orders that received at least one intervention:", (order_model.intervention_count > 0).sum())
print("3. Most common intervention:"); display(interventions.intervention_type.value_counts().to_frame("count"))
print("4. Frustrated late journeys (support or cancel attempt) with NO intervention:",
      (frustrated.intervention_count == 0).sum(), "of", len(frustrated))
display(frustrated[frustrated.intervention_count == 0][cols].sort_values("delay_min", ascending=False).head(5))

1. Late orders with a support interaction: 258 of 843 (31%)
2. Orders that received at least one intervention: 430
3. Most common intervention:


,count
intervention_type,
DRIVER_REASSIGNMENT,155
RESTAURANT_CONTACT,116
PRIORITY_DISPATCH,95
CUSTOMER_CREDIT,64


4. Frustrated late journeys (support or cancel attempt) with NO intervention: 192 of 259


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
193,O00194,C0771,True,False,0,,delivered,1.0,38.34
736,O00737,C0824,True,False,0,,delivered,1.0,36.28
302,O00303,C0411,True,False,0,,delivered,1.0,33.16
210,O00211,C0629,True,False,0,,delivered,1.0,32.38
1234,O01235,C0772,True,False,0,,delivered,1.0,32.00


### Challenge 3 answer

1. **258 of 843 late orders (31%)** had a support interaction: either the customer opened support in the app or a
   ticket exists.
2. **430 orders** received at least one intervention.
3. **Driver reassignment** is the most common (155), followed by restaurant contact (116).
4. **192 of 259 frustrated late journeys (74%) got no intervention at all.** These customers reached out or tried to
   cancel, and nobody did anything visible on the order.

Aggregating actions and interventions to one row per order *before* joining matters. Joining the raw tables directly
would multiply order rows by the number of actions and inflate every rate.

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [11]:
# Example starting point:
# valid_outcomes=outcomes[outcomes.late_flag.notna()]
# late_delivery_rate=valid_outcomes.late_flag.mean()

# TODO: calculate your selected metrics

In [12]:
valid = order_model[order_model.late_flag.notna()]

life = base_orders[["order_id", "created_at", "pickup_at", "actual_delivery_at"]].merge(
    dispatch[["order_id", "estimated_pickup_at"]], on="order_id")
for c in ["created_at", "pickup_at", "actual_delivery_at", "estimated_pickup_at"]:
    life[c] = pd.to_datetime(life[c], format="mixed")
life["pickup_slip_min"] = (life.pickup_at - life.estimated_pickup_at).dt.total_seconds() / 60
life = life.merge(outcomes[["order_id", "late_flag"]], on="order_id")

ivt = interventions.merge(life[["order_id", "pickup_at"]], on="order_id")
ivt["before_pickup"] = pd.to_datetime(ivt.intervention_at, format="mixed") < ivt.pickup_at
at_risk = life[life.pickup_slip_min > 5].order_id           # already running late at pickup
proactive = ivt[ivt.before_pickup].order_id.unique()

metrics = pd.DataFrame([
    ["Late delivery rate", "outcome", "late orders / delivered orders with a time", "order",
     f"{100 * valid.late_flag.mean():.1f}%", "This IS the project KPI"],
    ["Pickup slip", "workflow driver", "median(pickup_at - dispatch estimated_pickup_at)", "order",
     f"{life.pickup_slip_min.median():.1f} min (late orders {life[life.late_flag == 1].pickup_slip_min.median():.1f}, "
     f"on time {life[life.late_flag == 0].pickup_slip_min.median():.1f})",
     "The stage where late orders fall behind; reduce it and the KPI moves"],
    ["Support contact rate on late orders", "customer interaction", "late orders with support / late orders",
     "order", f"{100 * late.support_opened.mean():.1f}%", "Cost and customer pain caused by each late order"],
    ["Proactive intervention coverage", "intervention", "orders slipping >5 min at pickup that got an intervention before pickup / all such orders",
     "order", f"{100 * at_risk.isin(proactive).mean():.1f}%", "Only interventions before pickup can prevent lateness"],
    ["Unrecovered frustration", "customer interaction + intervention", "frustrated late orders with no intervention / frustrated late orders",
     "order", f"{100 * (frustrated.intervention_count == 0).mean():.1f}%", "Service recovery gap, even if lateness wasn't prevented"],
], columns=["metric", "type", "formula", "grain", "value", "link to KPI"])
metrics

,metric,type,formula,grain,value,link to KPI
0,Late delivery rate,outcome,late orders / delivered orders with a time,order,56.4%,This IS the project KPI
1,Pickup slip,workflow driver,median(pickup_at - dispatch estimated_pickup_at),order,"8.1 min (late orders 13.9, on time 1.8)",The stage where late orders fall behind; reduce it and the KPI moves
2,Support contact rate on late orders,customer interaction,late orders with support / late orders,order,30.6%,Cost and customer pain caused by each late order
3,Proactive intervention coverage,intervention,orders slipping >5 min at pickup that got an intervention before pickup / all such orders,order,19.6%,Only interventions before pickup can prevent lateness
4,Unrecovered frustration,customer interaction + intervention,frustrated late orders with no intervention / frustrated late orders,order,74.1%,"Service recovery gap, even if lateness wasn't prevented"


### Challenge 4 answer

I chose five metrics that follow the order journey:

- **Late delivery rate** (outcome, 56.4%). This is the KPI itself.
- **Pickup slip** (workflow driver): late orders are picked up ~14 minutes behind dispatch's plan, compared with ~2
  minutes for on-time orders. This is the most actionable lever.
- **Support contact rate on late orders** (customer interaction): shows what lateness costs in customer effort.
- **Proactive intervention coverage** (intervention): of the orders already slipping more than 5 minutes at pickup,
  only about 20% got an intervention before pickup, when it could still matter.
- **Unrecovered frustration** (74%): measures the service-recovery gap separately from the lateness itself.

Every metric is at order grain, so they can be joined and compared without double counting.

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [13]:
# TODO: use the order-level model plus joins/groupby
# Reminder: association != causation

In [14]:
print("A. Support interaction vs delay")
display(valid.groupby("support_opened").agg(orders=("order_id", "count"), late_rate=("late_flag", "mean"),
                                            median_delay=("delay_min", "median")).round(3))

print("B. Late rate with vs without intervention")
display(valid.groupby(valid.intervention_count > 0).agg(orders=("order_id", "count"), late_rate=("late_flag", "mean"),
                                                        median_delay=("delay_min", "median"))
        .rename_axis("had_intervention").round(3))

print("C. Late rate by intervention type, and when it happened")
by_type = (ivt.merge(outcomes[["order_id", "late_flag", "delay_min"]], on="order_id").dropna(subset=["late_flag"])
           .groupby("intervention_type").agg(orders=("order_id", "nunique"), late_rate=("late_flag", "mean"),
                                             share_before_pickup=("before_pickup", "mean"))
           .sort_values("late_rate").round(3))
display(by_type)

print("D. Restaurants with the most late orders")
r = (valid.groupby("restaurant_id").agg(orders=("order_id", "count"), late_orders=("late_flag", "sum"),
                                        late_rate=("late_flag", "mean"))
     .sort_values("late_orders", ascending=False).round(2))
display(r.head(5))
print(f"Top 10 restaurants account for {100 * r.late_orders.head(10).sum() / r.late_orders.sum():.0f}% of late orders")

print("E. Support + intervention + still late:",
      ((valid.late_flag == 1) & valid.support_opened & (valid.intervention_count > 0)).sum(), "orders")

A. Support interaction vs delay


,orders,late_rate,median_delay
support_opened,,,
False,1203,0.486,-0.250
True,292,0.884,12.285


B. Late rate with vs without intervention


,orders,late_rate,median_delay
had_intervention,,,
False,1091,0.564,1.39
True,404,0.564,1.54


C. Late rate by intervention type, and when it happened


,orders,late_rate,share_before_pickup
intervention_type,,,
PRIORITY_DISPATCH,86,0.512,0.663
RESTAURANT_CONTACT,108,0.528,0.602
DRIVER_REASSIGNMENT,147,0.578,0.748
CUSTOMER_CREDIT,63,0.667,0.016


D. Restaurants with the most late orders


,orders,late_orders,late_rate
restaurant_id,,,
R050,35,22.0,0.63
R004,30,20.0,0.67
R024,25,20.0,0.80
R023,32,20.0,0.62
R030,38,20.0,0.53


Top 10 restaurants account for 23% of late orders
E. Support + intervention + still late: 67 orders


### Challenge 5 answers

**A. Support vs delay.** Orders with a support interaction are late 88% of the time (median +12 min), against 49% for
the rest. *Tells us:* support contact is a strong signal of a bad delivery. *Does not prove:* that lateness causes
contact in every case. Some customers contact support for other reasons, and the direction is partly obvious.

**B. With vs without intervention.** The late rate is essentially **the same (56.4% vs 56.4%)**. *Tells us:* as they are
used today, interventions don't show up in the outcome. *Does not prove:* that they don't work. They're aimed at orders
already in trouble, so they may be pulling a worse group back up to average.

**C. By type.** Priority dispatch has the lowest late rate (51%). Customer credit has the highest (67%), but it happens
after delivery in most cases. It's *compensation*, not *prevention*, so it can't reduce lateness by design. *Does not
prove:* that priority dispatch works better, because the types are aimed at different situations.

**D. Restaurants.** The top 10 restaurants account for about **23% of late orders**. Lateness is spread widely, not
concentrated in a few bad kitchens. *Does not prove:* that restaurants are to blame. Busy restaurants simply have more
orders to be late.

**E. Support + intervention + still late:** **67 orders**. These are the journeys where we acted and still failed. They
are the most useful cases to review one by one, to see *when* the intervention came relative to pickup.

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

### Challenge 6: project view

```
PROJECT KPI        Reduce late delivery rate (56.4% of delivered orders; 23.3% by more than 10 min)
     │
OUTCOME METRIC     Late delivery rate, and share > 10 min
     │
WORKFLOW METRICS   Pickup slip vs dispatch plan  ·  order-to-pickup minutes  ·  pickup-to-delivery minutes
                   ·  driver reassignment rate
     │
INTERVENTIONS      Preventive: restaurant contact, priority dispatch, driver reassignment (only if before pickup)
                   Recovery:   customer credit (after the fact)
     │
DATA / EVENTS      orders (created, promised, pickup, delivered) · dispatch API (assigned, reassigned, planned pickup)
                   · customer_app_actions · support_tickets · order_interventions · restaurant_status (snapshot)
```

1. **Controllable by operations:** pickup slip (restaurant contact, prep SLAs), assignment and reassignment, and how
   early interventions happen.
2. **Outcomes:** late delivery rate, share over 10 minutes, and support contact rate.
3. **The missing event that limits the model most:** *driver arrived at restaurant*, together with a system-recorded
   *food ready* time. Without them, pickup slip can't be split into "kitchen slow" and "driver late", and that is the
   decision that says whether to fix restaurants or dispatch.
4. **Instrument next:** a geofenced arrival event, a restaurant "food ready" button with timestamps kept as history,
   and the trigger time on every intervention so we can tell prevention from recovery.

**Final deliverable summary**
- **Core entities:** customer, order, restaurant, driver.
- **Key events:** created, assigned, reassigned, picked up, delivered, app actions, tickets, interventions.
- **Relationships:** everything hangs off `order_id` (one-to-many for interactions and interventions, one-to-one for outcome).
- **Metrics:** late delivery rate, pickup slip, support contact rate on late orders, proactive intervention coverage,
  unrecovered frustration.
- **KPI linkage:** pickup slip is the workflow metric that moves the late rate; proactive interventions are the lever.
- **Modelling limitations:** there's no arrival or food-ready event, so the model can locate *where* delay builds up
  but not *who* causes it. The interventions log and dispatch also disagree about reassignments (only 8 overlap), so
  intervention effects should be read with care.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.